# Example queries: `report` (comstock_oedi)

Auto-generated from `tests/query_snapshots/report.json`. Each cell
runs one entry from the snapshot suite. Regenerate by running the
matching test with `--update-snapshot` or `--overwrite-snapshot`.


In [ ]:
from pathlib import Path
from buildstock_query import BuildStockQuery
import pandas as pd


## Construct the BuildStockQuery object

`cache_folder` points at the snapshot test cache directory so this
notebook reuses parquets that the test suite has already downloaded
from Athena. Queries that are already cached return immediately;
anything new still hits Athena.


In [ ]:
# This notebook lives in `tests/example_notebooks/`; the snapshot test
# cache is its sibling `tests/query_snapshots/comstock_oedi_cache/`. Resolve
# the path relative to the notebook directory (`_dh[0]` is set by
# IPython at kernel startup; falls back to CWD outside Jupyter).
_NB_DIR = Path(_dh[0] if "_dh" in globals() else ".").resolve()
_CACHE = (_NB_DIR / "../query_snapshots/comstock_oedi_cache").resolve()
bsq = BuildStockQuery(
    "rescore",
    "buildstock_sdr",
    "comstock_amy2018_r2_2025",
    buildstock_type="comstock",
    db_schema="comstock_oedi_state_and_county",
    skip_reports=True,
    cache_folder=str(_CACHE),
)


## `report_buildings_by_change_ok`

report.get_buildings_by_change for upgrade 1, change_type='ok-chng' — buildings whose upgrade succeeded with an acceptable energy change. Pins the change-condition SQL shape now that the OEDI applicability fix lets the report layer resolve.


In [ ]:
result = bsq.report.get_buildings_by_change(upgrade_id='1', change_type='ok-chng')
result.head() if hasattr(result, 'head') else result


# shape: (4257098, 5)
 bldg_id in.nhgis_tract_gisjoin state  applicability  applicability_1
    6929         G0201950000200    AK           True             True
       5         G0100690040202    AL           True             True
       6         G0100690040400    AL           True             True
       7         G0100690040600    AL           True             True
       9         G0100690041100    AL           True             True


## `report_successful_simulation_count_co`

report.get_successful_simulation_count restricted to CO. Pins the metadata-only count shape. Method was broken before fix in 2026-04-26: the call passed `bs_only=True` to _add_restrict() which doesn't accept that kwarg, causing TypeError on every call with a non-empty restrict.


In [ ]:
result = bsq.report.get_successful_simulation_count(restrict=[('state', ['CO'])])
result.head() if hasattr(result, 'head') else result


# shape: (1, 1)
 count
134649


## `report_successful_simulation_count_no_restrict`

report.get_successful_simulation_count with no restrict — pins the no-WHERE branch (only the implicit applicability=true filter). Pairs with report_successful_simulation_count_co to cover both restrict and no-restrict paths.


In [ ]:
result = bsq.report.get_successful_simulation_count()
result.head() if hasattr(result, 'head') else result


# shape: (1, 1)
  count
7602151


## `report_success_report`

report.get_success_report — returns three SQL statements (baseline counts, per-upgrade counts, change-type breakdown). Joined into one snapshot via the harness's MULTI_SQL_SEPARATOR. Pins the most-used reporting entry point. Requires the OEDI applicability fix in _rename_completed_status_column (commit added a lowercase-string normalize so True/False booleans map correctly).


In [ ]:
result = bsq.report.get_success_report(trim_missing_bs='auto')
result.head() if hasattr(result, 'head') else result


# shape: (62, 17)
  success  inapplicable  fail       Sum  Applied %  no-chng  bad-chng   ok-chng  true-bad-chng  true-ok-chng  null       any  no-chng %  bad-chng %  ok-chng %  true-ok-chng %  true-bad-chng %
7602151.0           0.0     0 7602151.0        0.0      0.0       0.0       0.0            0.0           0.0   0.0       0.0        0.0         0.0        0.0             0.0              0.0
4257229.0     3344922.0     0 7602151.0       56.0      0.0     131.0 4257098.0          131.0     4257098.0   0.0 4257229.0        0.0         0.0      100.0           100.0              0.0
4257064.0     3345087.0     0 7602151.0       56.0      0.0       0.0 4257064.0            0.0     4257064.0   0.0 4257064.0        0.0         0.0      100.0           100.0              0.0
4053004.0     3549147.0     0 7602151.0       53.3      0.0      37.0 4052967.0           37.0     4052967.0   0.0 4053004.0        0.0         0.0      100.0           100.0              0.0
4052904.0     3549247.

## `report_check_ts_bs_integrity`

report.check_ts_bs_integrity — every BuildStockQuery(skip_reports=False) init runs this. Pins the SQL shape of all queries it submits: per-upgrade ts distinct counts, metadata distinct counts (baseline + per-upgrade), duplicate-row checks, rows-per-building. Marked nondeterministic to skip data check — _get_rows_per_building scans the full TS table and would be a multi-TB landmine to actually run. SQL-shape regression coverage for the comma-join class of bug fixed in commit 9cd148d (md_table column refs mixed with bs_table predicates produced FROM x, x AS bs Cartesians that Athena rejected with 'mismatched input GROUP').


In [ ]:
result = bsq.report.check_ts_bs_integrity()
result.head() if hasattr(result, 'head') else result
